In [101]:
# For later: RAD - https://github.com/ubc-systopia/dsn-2022-rad-artifact/tree/main

import pandas as pd
import glob
import os

data_paths = ["../data/rad/known_procedures/benign","../data/rad/known_procedures/anomaly", "../data/rad/unknown_procedures"]

files = []
for data_path in data_paths:
    files += sorted(glob.glob(os.path.join(data_path, "*.csv")))


# files = sorted(glob.glob(os.path.join(data_paths[0], "*.csv")))

# Load and concatenate
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)


In [102]:
df.head()

,Timestamp,Module,Method_Name,Arguments,Responses,Exceptions,id,Execution Time (Sec),Arrival_Time,Departure_Time
0,2021:10:12:13:22:23.845243,C9,_init_,ftdi: None,NaN,NaN,NaN,NaN,NaN,NaN
1,2021:10:12:13:22:24.499940,C9,PING,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2021:10:12:13:22:24.799885,C9,BIAS,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2021:10:12:13:22:25.072611,C9,SPED,"velocity: 20000, acceleration: 20000",NaN,NaN,NaN,NaN,NaN,NaN
4,2021:10:12:13:22:25.453128,C9,BIAS,bias: 0,NaN,NaN,NaN,NaN,NaN,NaN


In [103]:
df.drop(columns=['Module', 'Responses', "Exceptions", 'Arrival_Time', 'Departure_Time', 'Execution Time (Sec)', 'id'], inplace=True)
df.head()

,Timestamp,Method_Name,Arguments
0,2021:10:12:13:22:23.845243,_init_,ftdi: None
1,2021:10:12:13:22:24.499940,PING,NaN
2,2021:10:12:13:22:24.799885,BIAS,NaN
3,2021:10:12:13:22:25.072611,SPED,"velocity: 20000, acceleration: 20000"
4,2021:10:12:13:22:25.453128,BIAS,bias: 0


In [104]:
df = df[df["Method_Name"].str.contains("ARM", case=False, na=False)]
df = df[~df["Arguments"].str.contains("velocity", case=False, na=False)]
df.drop(columns=['Method_Name'], inplace=True)
df.reset_index(drop=True, inplace=True)

df

,Timestamp,Arguments
0,2021:10:12:14:35:31.590194,"X: 159525, Y: 182500, Z: 187000, gripper: 1089..."
1,2021:10:12:14:35:32.141240,"X: 160151, Y: 182500, Z: 187000, gripper: 1089..."
2,2021:10:12:14:35:32.717996,"X: 160776, Y: 182500, Z: 187000, gripper: 1089..."
3,2021:10:12:14:35:33.188219,"X: 161401, Y: 182500, Z: 187000, gripper: 1089..."
4,2021:10:12:14:35:33.536488,"X: 162027, Y: 182500, Z: 187000, gripper: 1089..."
...,...,...
11098,2021:10:22:14:57:07.749848,"X: 252600, Y: -123400, Z: 290070, gripper: 785..."
11099,2021:10:22:14:57:07.749848,"X: 252600, Y: -123400, Z: 290070, gripper: 785..."
11100,2021:10:22:14:57:07.749848,"X: 252600, Y: -123400, Z: 290070, gripper: 785..."
11101,2021:10:22:14:57:07.749848,"X: 252600, Y: -123400, Z: 290070, gripper: 785..."


In [105]:
# Encoding timestamps

# Convert to datetime
df['Timestamp'] = df['Timestamp'].astype(str).str.strip('"').str.strip("'").str.strip()

# Clean and normalize timestamp format
df['Timestamp'] = (
    df['Timestamp']
    .astype(str)
    .str.strip('"')
    .str.strip("'")
    .str.strip()
    .str.replace(":", "-", 2)     # Replace first two colons (Y:M:D → Y-M-D)
    .str.replace(":", "T", 1)     # Replace next colon (between day and hour) with T
)

df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='ISO8601', utc=True)

# Extract components, only use hour or shorter since all done in a single day
df['hour'] = df['Timestamp'].dt.hour
df['minute'] = df['Timestamp'].dt.minute
df['second'] = df['Timestamp'].dt.second
df['microsecond'] = df['Timestamp'].dt.microsecond

df.drop(columns=['Timestamp'], inplace=True)

df['time'] = (
    df['hour'] * 3600 + df['minute'] * 60 + df['second'] + df['microsecond'] / 1_000_000
)
df = df.sort_values('time')
df['time'] =  df['time'] - df['time'].iloc[0]

df.drop(columns=['hour', 'minute', 'second', 'microsecond'], inplace=True)

df.reset_index(drop=True, inplace=True)

df

,Arguments,time
0,"X: -186053, Y: -13900, Z: 250000, gripper: 150...",0.000000
1,"X: -181287, Y: -4773, Z: 250000, gripper: 1506...",0.846160
2,"X: -174021, Y: 2766, Z: 250000, gripper: 15066...",1.290711
3,"X: -165193, Y: 8004, Z: 250000, gripper: 15066...",1.717405
4,"X: -156365, Y: 13004, Z: 250000, gripper: 1506...",2.163429
...,...,...
11098,"X: -154900, Y: 212000, Z: 250000, gripper: 150...",16532.287049
11099,"X: -144900, Y: 212714, Z: 250000, gripper: 150...",16532.646601
11100,"X: -134900, Y: 213429, Z: 250000, gripper: 150...",16533.064527
11101,"X: -124900, Y: 214619, Z: 250000, gripper: 150...",16533.480524
